# Multimodal Cancer Classification Challenge 2026 — Improved

**Improvements over ResNet-18 baseline:**
- **EfficientNet-B0** backbone (timm) — stronger features
- **D4 augmentation** — random 90° steps (cells have no canonical orientation)
- **Stronger ColorJitter** (0.3) + **RandomErasing** (p=0.2)
- **Mixup** (α=0.2) and **label smoothing** (ε=0.05)
- **OneCycleLR** with 10 % warmup + **gradient clipping**
- **Early stopping** (patience=6) to avoid overfitting
- **DataParallel** (auto) for dual-T4
- **8-way D4 TTA** at inference (4 rotations × 2 reflections)

**Settings to enable before running:**
- Accelerator = **GPU T4 x2** (or P100)
- Internet = **On** (needed for ImageNet weights + timm)
- Persistence = **Files only**

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"   # force live stdout in Kaggle commits

import re, json, time, random, glob, sys, functools
from pathlib import Path

# Make every print() flush immediately so commit logs show progress in real time.
print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, LeaveOneGroupOut
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
print("timm:", timm.__version__)
!nvidia-smi -L

In [2]:
DATA_ROOT = Path("/kaggle/input/datasets/rafaelproena/a3-adl")
assert (DATA_ROOT / "train.csv").exists(), f"train.csv not at {DATA_ROOT}"
print("DATA_ROOT =", DATA_ROOT)
print("Contents:", sorted(p.name for p in DATA_ROOT.iterdir()))

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CV           = "sgkf"           # sgkf | gkf | lopo
N_SPLITS     = 3
SEED         = 1
EPOCHS       = 15
PATIENCE     = 6                # early stopping
BATCH_SIZE   = 96               # lowered from 128 to ease RAM pressure
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MIXUP_ALPHA  = 0.2
LABEL_SMOOTH = 0.05
GRAD_CLIP    = 1.0
DROPOUT      = 0.4
BACKBONE     = "efficientnet_b0"  # or "resnet18" for quick tests
NUM_WORKERS  = 2                # match Kaggle's 2 CPUs (was 4 → caused OOM)
PRETRAINED   = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

DATA_ROOT = /kaggle/input/datasets/rafaelproena/a3-adl
Contents: ['BF', 'FL', 'sampleSubmission.csv', 'train.csv']


In [3]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

class CellDataset(Dataset):
    def __init__(self, df, bf_dir, fl_dir, bf_transform, fl_transform, paired_transform=None):
        self.df = df.reset_index(drop=True)
        self.bf_dir = Path(bf_dir); self.fl_dir = Path(fl_dir)
        self.bf_transform = bf_transform; self.fl_transform = fl_transform
        self.paired_transform = paired_transform

    def __len__(self): return len(self.df)

    @staticmethod
    def _load(p): return Image.open(p).convert("L")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_transform(self._load(self.bf_dir / name))
        fl = self.fl_transform(self._load(self.fl_dir / name))
        if self.paired_transform is not None:
            bf, fl = self.paired_transform(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

In [4]:
def stratified_patient_kfold(df, n_splits=3, seed=1, strict=True):
    skgf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df["Diagnosis"].to_numpy(); groups = df["patient_id"].to_numpy()
    splits = list(skgf.split(df, y=y, groups=groups))
    if strict:
        for f, (_, va) in enumerate(splits):
            if len(np.unique(y[va])) < 2:
                raise ValueError(f"Fold {f} has only one class — try a different SEED.")
    return splits

def leave_one_patient_out(df):
    return list(LeaveOneGroupOut().split(df, groups=df["patient_id"].to_numpy()))

def get_splits(df, cv, n_splits, seed):
    if cv == "sgkf": return stratified_patient_kfold(df, n_splits, seed)
    if cv == "gkf":  return list(GroupKFold(n_splits=n_splits).split(df, groups=df["patient_id"]))
    if cv == "lopo": return leave_one_patient_out(df)
    raise ValueError(cv)

def summarize_split(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val: {len(va):>6} cells, {vad['patient_id'].nunique():>2}p, "
            f"pos {vad['Diagnosis'].mean():.3f} | "
            f"val pats: {sorted(vad['patient_id'].unique().tolist())}")

In [5]:
# Normalization helpers
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)


class PairedGeoAug:
    """Identical geometric augmentation applied to both BF and FL tensors.

    Applies (1) a random 90-degree step, (2) random flip(s), (3) fine rotation.
    Cells have no canonical orientation, so the 90-degree steps matter.
    """
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=15.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot

    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        return bf, fl


def train_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([
        T.ColorJitter(brightness=0.3, contrast=0.3),
        norm,
        T.RandomErasing(p=0.2, scale=(0.02, 0.15), value=0),
    ])


def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

In [6]:
def _make_effnet_b0_branch(pretrained=True):
    """EfficientNet-B0 (timm), 1-channel input. Returns (module, feat_dim=1280)."""
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                             num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                         stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280


def _make_resnet18_branch(pretrained=True):
    """ResNet-18, 1-channel input. Returns (module, feat_dim=512)."""
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd  # 512


def _make_branch(backbone, pretrained):
    if backbone == "efficientnet_b0": return _make_effnet_b0_branch(pretrained)
    return _make_resnet18_branch(pretrained)


class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT, backbone=BACKBONE):
        super().__init__()
        self.bf_branch, fd = _make_branch(backbone, pretrained)
        self.fl_branch, _  = _make_branch(backbone, pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)


# Quick sanity check: forward pass with random input
with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    print("Model output shape:", _m(_x, _x).shape,
          "  params:", sum(p.numel() for p in _m.parameters()) // 1_000_000, "M")
    del _m, _x

Model output shape: torch.Size([2])   params: 9 M


In [7]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
per_pat = df_train.groupby("patient_id").agg(
    n_cells=("Name", "size"), label=("Diagnosis", "first"),
).reset_index()
print(per_pat.to_string(index=False))
print(f"\nTest: {len(df_test)} cells")

Train: 114302 cells, 12 patients, pos rate 0.3876
 patient_id  n_cells  label
          3    10000      1
          5     4302      1
          7    10000      0
          9    10000      0
         10    10000      0
         11    10000      0
         13    10000      0
         14    10000      0
         15    10000      0
         16    10000      1
         17    10000      1
         18    10000      1

Test: 59040 cells


In [ ]:
def mixup_batch(bf, fl, y, alpha=0.2):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(bf.size(0), device=bf.device)
    return (lam * bf + (1 - lam) * bf[idx],
            lam * fl + (1 - lam) * fl[idx],
            lam * y  + (1 - lam) * y[idx])


def smooth_labels(y, eps=0.05):
    return y * (1.0 - eps) + eps * 0.5


def run_epoch(model, loader, optimizer, scaler, criterion, train,
              mixup_alpha=0.0, label_smooth=0.0, grad_clip=0.0, sched=None):
    model.train(train)
    losses, hard_ys, ps = [], [], []
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())  # un-mixed labels for AUC

        if train:
            if mixup_alpha > 0: bf, fl, y = mixup_batch(bf, fl, y, mixup_alpha)
            if label_smooth > 0: y = smooth_labels(y, label_smooth)

        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss = criterion(logits, y)

        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                
                if sched is not None: 
                    # Only step scheduler if optimizer.step() wasn't skipped due to inf/nan gradients
                    if scaler.get_scale() >= old_scale:
                        sched.step()
            else:
                loss.backward()
                if grad_clip > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if sched is not None: 
                    sched.step()  # OneCycleLR steps per batch

        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())

    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, hard_ys, ps


def train_fold(df, train_idx, val_idx, fold):
    bf_dir = DATA_ROOT / "BF" / "train"; fl_dir = DATA_ROOT / "FL" / "train"
    train_ds = CellDataset(df.iloc[train_idx], bf_dir, fl_dir,
                           train_modality_transform("bf"), train_modality_transform("fl"),
                           paired_transform=PairedGeoAug())
    val_ds   = CellDataset(df.iloc[val_idx], bf_dir, fl_dir,
                           eval_modality_transform("bf"), eval_modality_transform("fl"))

    # persistent_workers=False so RAM is reclaimed between train/val phases
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=False)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=False)

    model = MultimodalClassifier(pretrained=PRETRAINED, dropout=DROPOUT,
                                 backbone=BACKBONE).to(DEVICE)
    if torch.cuda.device_count() > 1:
        print(f"  DataParallel x{torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    pos = (df.iloc[train_idx]["Diagnosis"] == 1).sum()
    neg = (df.iloc[train_idx]["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  backbone={BACKBONE}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        steps_per_epoch=len(train_loader), epochs=EPOCHS,
        pct_start=0.1,
    )
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    history, best_auc, no_improve = [], -1.0, 0
    ckpt_path = OUT_DIR / f"fold{fold}_best.pt"

    for ep in range(EPOCHS):
        t0 = time.time()
        tr_loss, tr_auc, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            mixup_alpha=MIXUP_ALPHA, label_smooth=LABEL_SMOOTH,
            grad_clip=GRAD_CLIP, sched=sched,
        )
        with torch.no_grad():
            va_loss, va_auc, vy, vp = run_epoch(
                model, val_loader, None, None, criterion, False
            )
        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| va_loss {va_loss:.4f} va_auc {va_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "va_auc": va_auc, "time": dt})

        if va_auc > best_auc:
            best_auc, no_improve = va_auc, 0
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save({"model": state, "epoch": ep, "val_auc": va_auc,
                        "args": {"backbone": BACKBONE, "dropout": DROPOUT}}, ckpt_path)
            oof = pd.DataFrame({"Name": df.iloc[val_idx]["Name"].values,
                                "patient_id": df.iloc[val_idx]["patient_id"].values,
                                "y_true": vy, "y_pred": vp})
            oof.to_csv(OUT_DIR / f"fold{fold}_oof.csv", index=False)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  Early stopping at epoch {ep}")
                break

    # Free GPU memory between folds (important when running all 3 folds in one cell)
    del model, optimizer, sched, scaler, train_loader, val_loader
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    with open(OUT_DIR / f"fold{fold}_history.json", "w") as f:
        json.dump({"history": history, "best_auc": best_auc}, f, indent=2)
    return best_auc

In [9]:
# Train all folds — ~12-18 min per fold on T4 with EfficientNet-B0, EPOCHS=15
splits = get_splits(df_train, CV, N_SPLITS, SEED)
print(f"CV={CV}, folds={len(splits)}  backbone={BACKBONE}\n")

best_aucs = []
for fold, (tr, va) in enumerate(splits):
    print(f"=== FOLD {fold} ===")
    print("  " + summarize_split(df_train, tr, va))
    best = train_fold(df_train, tr, va, fold)
    best_aucs.append(best)
    print(f"  best AUC fold {fold} = {best:.4f}\n")

print(f"Per-fold best AUC: {[f'{a:.4f}' for a in best_aucs]}")
print(f"Mean: {np.mean(best_aucs):.4f}  Std: {np.std(best_aucs):.4f}")

CV=sgkf, folds=3  backbone=efficientnet_b0

=== FOLD 0 ===
  train:  74302 cells,  8p, pos 0.327 | val:  40000 cells,  4p, pos 0.500 | val pats: [3, 9, 14, 18]


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  DataParallel x2 GPUs
  pos_weight=2.057  backbone=efficientnet_b0


/tmp/ipykernel_23/2282575668.py:44: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  if sched is not None: sched.step()  # OneCycleLR steps per batch


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for hp in sorted(glob.glob(str(OUT_DIR / "fold*_history.json"))):
    h = json.load(open(hp))["history"]
    label = Path(hp).stem.replace("_history", "")
    ax[0].plot([e["epoch"] for e in h], [e["va_loss"] for e in h], marker="o", label=label)
    ax[1].plot([e["epoch"] for e in h], [e["va_auc"]  for e in h], marker="o", label=label)
ax[0].set(title="Validation loss", xlabel="epoch", ylabel="BCE")
ax[1].set(title="Validation AUC",  xlabel="epoch", ylabel="AUC")
ax[1].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="target 0.85")
for a in ax: a.legend(); a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
oofs = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(str(OUT_DIR / "fold*_oof.csv")))])
print(f"OOF AUC overall: {roc_auc_score(oofs['y_true'], oofs['y_pred']):.4f}  on {len(oofs)} cells")

print("\nPer-patient mean prediction vs true label:")
pp = oofs.groupby("patient_id").agg(
    mean_pred=("y_pred", "mean"), median_pred=("y_pred", "median"),
    label=("y_true", "first"),
).sort_values("mean_pred")
print(pp.to_string())
print(f"\nPatient-level AUC: {roc_auc_score(pp['label'], pp['mean_pred']):.4f}")

In [ ]:
def _d4_augments(bf, fl):
    """Yield all 8 elements of D4 (4 rotations × 2 reflections)."""
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)


def predict_one_ckpt(ckpt_path, loader, tta=True):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    saved = state.get("args", {})
    backbone = saved.get("backbone", "resnet18")
    dropout  = float(saved.get("dropout", 0.3))
    model = MultimodalClassifier(pretrained=False, backbone=backbone,
                                 dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    preds = []
    n_aug = 8 if tta else 1
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in (_d4_augments(bf, fl) if tta else [(bf, fl)]):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    return np.concatenate(preds)


test_ds = CellDataset(df_test, DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test",
                      eval_modality_transform("bf"), eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

ckpts = sorted(glob.glob(str(OUT_DIR / "fold*_best.pt")))
print("Using checkpoints:", ckpts)
all_preds = []
for ckpt in ckpts:
    print(f"  - {ckpt}")
    all_preds.append(predict_one_ckpt(ckpt, test_loader, tta=True))
preds = np.mean(all_preds, axis=0)

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f})")
print(sub.head())